# 18 · App-server：接受请求与完成任务是两件事

服务入口把多客户端接到同一内核，但不能根据 RPC 返回自行制造业务完成状态。
本章专注协议关联与反向请求：初始化、相同数字 ID 的不同连接、指定接收者、单次结算和断连清理。
我们真的启动一个临时 127.0.0.1 TCP 服务并发送逐行 JSON；这是教学传输，不是 Codex 的 WebSocket，
也没有真实审批策略或 Agent Scheduler。服务只处理 echo 与合成 input/ask。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [E01 · JSONRPCMessage / RequestId](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/app-server-protocol/src/rpc.rs#L1-L88)：信封不要求或发送 jsonrpc:2.0；定义 request/notification/response/error，ID 为字符串或 i64。
- [E05 · notify_client_response / take_connection_callback](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/app-server/src/outgoing_message.rs#L563-L685)：回答经 callback 取走后单次唤醒；HumanCommand 检查能力和 recipients；verification 检查 owner/身份代次；普通请求没有同样 recipient 检查。
- [E07 · turn_start_inner](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/app-server/src/request_processors/turn_processor.rs#L719-L801)：turn/start 调用现有 thread.start_or_steer_turn，处理 started/steered/not_submitted，返回 InProgress。


## 实验 1 · 每个连接单独初始化


初始化状态属于连接，不能被别的客户端借用。先发送未初始化请求，观察拒绝，再初始化。
本 helper 只负责发帧/读帧，2 秒超时用于暴露挂起，不自动重试。


In [2]:
rpc = load_example('18', 'relay')
import asyncio, json
relay = rpc.Relay()
address = await relay.start()
async def send(client, frame):
    client['writer'].write(json.dumps(frame).encode()+b'\n')
    await client['writer'].drain()
async def receive(client):
    raw = await asyncio.wait_for(client['reader'].readline(), 2)
    assert raw, 'unexpected connection close'
    return json.loads(raw)
async def request(client, frame):
    await send(client, frame)
    return await receive(client)
async def connect():
    reader, writer = await asyncio.open_connection(*address)
    return {'reader':reader, 'writer':writer}
async def initialize(client):
    answer = await request(client, {'id':'init','method':'initialize'})
    client['id'] = answer['result']['peerId']
    return client
A = await connect()
rejected = await request(A, {'id':1,'method':'protocol/echo','params':{'x':1}})
assert rejected['error']['message'] == 'Not initialized'
await initialize(A)
B, C = await initialize(await connect()), await initialize(await connect())
assert len({A['id'], B['id'], C['id']}) == 3
again = await request(A, {'id':'init-again','method':'initialize'})
assert again['error']['message'] == 'Already initialized'
print({'transport':'loopback TCP', 'port':address[1], 'initialized_peers':[A['id'],B['id'],C['id']]})


{'transport': 'loopback TCP', 'port': 60362, 'initialized_peers': [1, 2, 3]}


## 实验 2 · 同一个 request ID 可以同时属于两个连接


连接 A、B 都用 id=77；返回应沿各自连接交付。请求 ID 解决“这是谁的回复”，不表达任务已经完成。
为了不伪造内核，本原型根本不提供 turn/start；未知方法必须报错，而不是成功回声。


In [3]:
answers = await asyncio.gather(
    request(A, {'id':77,'method':'protocol/echo','params':{'owner':'A'}}),
    request(B, {'id':77,'method':'protocol/echo','params':{'owner':'B'}}))
assert [a['id'] for a in answers] == [77,77]
assert [a['result']['owner'] for a in answers] == ['A','B']
unsupported = await request(C, {'id':2,'method':'turn/start','params':{}})
assert unsupported['error']['code'] == -32601
print({'connection_scoped_answers': answers, 'turn_start': unsupported['error']['message']})


{'connection_scoped_answers': [{'id': 77, 'result': {'owner': 'A'}}, {'id': 77, 'result': {'owner': 'B'}}], 'turn_start': 'Method not found'}


## 实验 3 · 反向问题只能被合格接收者结算一次


先向 A、B 发同一问题，再让 C 猜中 ID 回答，随后让 A 提交畸形答案。两次都不能消费 pending。
B 的有效回答才结算；A 的迟到重复回复不能再唤醒一次。教学版对全部问题检查 recipients，
固定源码对不同反向请求类别有不同规则，不能把加严教学策略当成原实现通则。


In [4]:
qid, future = await relay.ask({A['id'],B['id']}, {'question':'Synthetic: may read fixture?'})
a_frame, b_frame = await asyncio.gather(receive(A),receive(B))
assert a_frame['id'] == b_frame['id'] == qid
await send(C, {'id':qid,'result':{'answer':'not invited'}})
await request(C, {'id':'barrier-c','method':'protocol/echo','params':{}})
assert not future.done() and qid in relay.pending
await send(A, {'id':qid,'result':{'answer':42}})
await request(A, {'id':'barrier-a','method':'protocol/echo','params':{}})
assert not future.done() and qid in relay.pending
await send(B, {'id':qid,'result':{'answer':'allow read of synthetic fixture'}})
answer = await asyncio.wait_for(future,2)
assert answer['status'] == 'answered' and qid not in relay.pending
await send(A, {'id':qid,'result':{'answer':'late duplicate'}})
await request(A, {'id':'barrier-duplicate','method':'protocol/echo','params':{}})
assert sum(row[0]=='settled' and row[-1]==qid for row in relay.audit) == 1
print({'answer':answer, 'audit':relay.audit[-4:]})


{'answer': {'status': 'answered', 'answer': 'allow read of synthetic fixture'}, 'audit': [('rejected_response', 3, 'server:1'), ('invalid_answer', 1, 'server:1'), ('settled', 2, 'server:1'), ('rejected_response', 1, 'server:1')]}


## 实验 4 · 一个客户端断开，不取消别人的等待


问题的接收者集合随断连收缩；最后一个合格接收者离开后，得到 no_recipients，不能擅自批准。
第三个连接继续 echo，证明局部断连没有关掉整个服务。这里只清理协议等待，不声称取消真实内核任务。


In [5]:
qid2, pending = await relay.ask({A['id'],B['id']}, {'question':'Second synthetic question'})
await asyncio.gather(receive(A),receive(B))
closed_A = relay.peers[A['id']].closed
A['writer'].close()
await A['writer'].wait_closed()
await asyncio.wait_for(closed_A.wait(),2)
assert not pending.done() and relay.pending[qid2].recipients == {B['id']}
closed_B = relay.peers[B['id']].closed
B['writer'].close()
await B['writer'].wait_closed()
await asyncio.wait_for(closed_B.wait(),2)
assert await asyncio.wait_for(pending,2) == {'status':'no_recipients'}
alive = await request(C, {'id':'alive','method':'protocol/echo','params':{'still':'alive'}})
assert alive['result']['still'] == 'alive'
print({'orphaned_question':qid2, 'remaining_peer':C['id']})


{'orphaned_question': 'server:2', 'remaining_peer': 3}


## 实验 5 · 重连建立新身份，最后确认服务收尾


重新连接不是复用旧 socket 的身份，更不是自动重放旧业务调用。新问题可以重新建立等待，但旧问题 ID 仍无效。
最后关闭所有连接和监听器，确认 pending、peers 和服务任务均已清空。


In [6]:
D = await initialize(await connect())
try:
    assert D['id'] not in {A['id'], B['id']}
    await send(D, {'id':qid2,'result':{'answer':'stale'}})
    await request(D, {'id':'barrier-stale','method':'protocol/echo','params':{}})
    assert qid2 not in relay.pending
    fresh_id, fresh_future = await relay.ask({D['id']}, {'question':'Fresh request'})
    assert (await receive(D))['id'] == fresh_id
    await send(D, {'id':fresh_id,'result':{'answer':'fresh'}})
    assert (await asyncio.wait_for(fresh_future,2))['answer'] == 'fresh'
finally:
    C['writer'].close()
    D['writer'].close()
    await asyncio.gather(C['writer'].wait_closed(), D['writer'].wait_closed())
    await asyncio.wait_for(relay.close(),3)
assert relay.pending == {} and relay.peers == {} and relay.tasks == set()
print('新连接完成新请求；全部连接、pending 和后台服务任务已收尾。')


新连接完成新请求；全部连接、pending 和后台服务任务已收尾。


## 你现在可以解释什么

能区分连接 ID、请求 ID 与业务 thread/turn 身份，并解释响应、通知、反向请求如何由不同 owner 结算。这里未实现认证、真实审批、WebSocket、持久订阅恢复或任务执行。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
